# 01 — Exploratory Data Analysis

Quick look at collected Reddit posts: volumes, time distribution, engagement, hashtags.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import RAW_DIR, INTERIM_DIR

sns.set_theme(style='whitegrid')

In [ ]:
# Load raw posts
df = pd.read_parquet(RAW_DIR / 'posts.parquet')
print(f'Total posts: {len(df):,}')
df.head()

In [ ]:
# Posts per subreddit
by_sub = df.groupby('subreddit').size().sort_values(ascending=False)
ax = by_sub.plot(kind='barh', figsize=(8, 4), color='#1f77b4')
ax.set_title('Posts per subreddit')
ax.set_xlabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Posts over time
df['date'] = pd.to_datetime(df['created_utc'])
weekly = df.set_index('date').resample('W').size()
fig, ax = plt.subplots(figsize=(10, 4))
weekly.plot(ax=ax, color='#2ca02c')
ax.set_title('Posts per week')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Engagement distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['score'].clip(upper=df['score'].quantile(0.99)), ax=axes[0], bins=50)
axes[0].set_title('Score distribution (clipped at 99th pct)')
sns.histplot(df['num_comments'].clip(upper=df['num_comments'].quantile(0.99)), ax=axes[1], bins=50)
axes[1].set_title('Comments distribution (clipped at 99th pct)')
plt.tight_layout()
plt.show()

In [ ]:
# After preprocessing — token length
clean = pd.read_parquet(INTERIM_DIR / 'clean.parquet')
print(f'Clean docs: {len(clean):,}')
print(f'Avg tokens: {clean["n_tokens"].mean():.1f}')
print(f'Median tokens: {clean["n_tokens"].median():.0f}')
clean['n_tokens'].clip(upper=clean['n_tokens'].quantile(0.99)).hist(bins=50, figsize=(8, 4))
plt.title('Token count per document')
plt.xlabel('tokens')
plt.show()

In [ ]:
# Top hashtags
import re
all_tags = []
for txt in clean['clean_text'].dropna():
    all_tags.extend(re.findall(r'#\w+', txt.lower()))
tag_counts = pd.Series(all_tags).value_counts().head(20)
tag_counts.plot(kind='barh', figsize=(8, 6), color='#d62728')
plt.title('Top hashtags')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()